# 01 — Descriptors, Properties, and Metaclasses

Learn attribute lookup, reusable validating fields, method binding, and the limited use cases for dynamic class construction.

*Advanced · 90–150 min*

## Objectives
- Explain how properties implement descriptors
- Write __get__, __set__ and __set_name__ methods
- Understand bound methods
- Evaluate trade-offs of metaclasses vs ordinary classes

## Examples

### 1. Properties and derived state

A property can validate assignments or expose a calculated value without storing redundant fields. Avoid hiding costly I/O inside a property, where readers may expect a cheap attribute lookup.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius
    @property
    def celsius(self):
        return self._celsius
    @celsius.setter
    def celsius(self, value):
        self._celsius = float(value)
    @property
    def fahrenheit(self):
        return self.celsius * 9 / 5 + 32

value = Temperature(20)
value.celsius = 25
print(value.fahrenheit)

**Design question:** Would a derived fahrenheit field risk becoming stale if stored directly?

### 2. Data descriptors for reusable validation

Descriptors belong to the class, but their per-instance data should be stored on each instance. __set_name__ gives the descriptor its attribute name when the owning class is created. Avoid shared state held on a descriptor.

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = name
        self.private = "_" + name
    def __get__(self, obj, owner):
        return self if obj is None else obj.__dict__[self.private]
    def __set__(self, obj, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f"{self.name} must be positive")
        obj.__dict__[self.private] = value

class Product:
    price = Positive()
    def __init__(self, price):
        self.price = price

first, second = Product(5), Product(9)
first.price = 7
print(first.price, second.price)

**Design question:** Why would assigning to self.value inside Positive.__set__ be wrong?

### 3. Method binding

When a function is accessed as an instance method, its descriptor machinery binds self. Access through the class leaves the instance argument explicit. This explains method signatures and monkey-patching behavior.

In [ ]:
class Greeter:
    def hello(self, name):
        return f"Hello, {name}"

greeter = Greeter()
print(greeter.hello("Ada"))
print(Greeter.hello(greeter, "Ada"))
print(greeter.hello.__self__ is greeter)

**Design question:** What is the difference between a bound method and the underlying function?

### 4. Dynamic classes and restraint

type(name, bases, namespace) creates a class dynamically. This can help frameworks but should not replace straightforward declarations. Prefer class decorators, factories, or a normal class when possible.

In [ ]:
def describe(self):
    return f"Plugin: {self.name}"

Plugin = type("Plugin", (), {"describe": describe})
instance = Plugin()
instance.name = "serializer"
print(instance.describe(), isinstance(instance, Plugin))

**Design question:** What complexity does this dynamic example introduce compared with class Plugin: ...?

## Exercises

**Exercise 1:** Write a NonNegative descriptor with an independent value for each instance.

In [ ]:
# Exercise 1
# TODO: Implement, demonstrate, and add assertions.

**Exercise 2:** Add a read-only computed property to a dataclass-like object.

In [ ]:
# Exercise 2
# TODO: Implement, demonstrate, and add assertions.

**Exercise 3:** Explain __get__ behavior when an attribute is accessed via a class.

In [ ]:
# Exercise 3
# TODO: Implement, demonstrate, and add assertions.

**Exercise 4:** Refactor a dynamically generated class into an ordinary class.

In [ ]:
# Exercise 4
# TODO: Implement, demonstrate, and add assertions.

## Common mistakes

- Store per-object data on instances, not on shared descriptors.
- Class construction magic rarely belongs in ordinary application code.
- Properties should not hide unexpected network or disk activity.

**Next:** [02 — Object References, Copying, Garbage Collection, and Weakrefs](02_memory_references_weakrefs.ipynb)